<a href="https://colab.research.google.com/github/Zafeerxyz/Machine-Learning-algorithms/blob/main/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
import joblib

df= sns.load_dataset("titanic")

X = df[['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']]
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

numeric_features = ['age','fare','sibsp','parch']
categorical_features = ['pclass','sex','embarked']

numeric_transformer = Pipeline(steps=[('imputer',SimpleImputer(strategy='median')),
                                      ("scaler",StandardScaler())
                                      ])
categorical_transformation = Pipeline(steps = [("encoder",OneHotEncoder(handle_unknown = 'ignore')),
                                               ("imputer",SimpleImputer(strategy = "most_frequent"))
                                               ])
preprocessor= ColumnTransformer(transformers=[('num',numeric_transformer,numerical_features),
                                              ("cat",categorical_transformation,categorical_features)])

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

full_pipeline = Pipeline(steps= [('preprocessor',preprocessor),
                                 ('classifier',RandomForestClassifier(random_state=42))
                                 ])

param_grid= {
    'classifier__n_estimators':[50,100],
    'classifier__max_depth':[None,5,10],
    'classifier__min_samples_split':[2,5]
}

grid_search = GridSearchCV(estimator=full_pipeline,param_grid=param_grid,cv=5,scoring='accuracy',n_jobs =-1)

grid_search.fit(X_train,y_train)

print("Best parameter:",grid_search.best_params_)
print("Best cv",grid_search.best_score_)

from sklearn.metrics import accuracy_score,classification_report
y_pred = grid_search.predict(X_test)

test_accuracy = accuracy_score(y_test,y_pred)

print("Final test",test_accuracy)

print("classification")
print(classification_report(y_test,y_pred))

joblib.dump(grid_search.best_estimator_, 'titanic_model_pipeline.pkl')

loaded_model = joblib.load('titanic_model_pipeline.pkl')
new_passenger = pd.DataFrame({
    'pclass': [3],
    'sex': ['male'],
    'age': [22],
    'sibsp': [0],
    'parch': [0],
    'fare': [7.25],
    'embarked': ['S']
})

prediction = loaded_model.predict(new_passenger)
print(f"Prediction for new passenger: {'Survived' if prediction[0] == 1 else 'Did not survive'}")



Best parameter: {'classifier__max_depth': 5, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 50}
Best cv 0.8258248793460062
Final test 0.7932960893854749
classification
              precision    recall  f1-score   support

           0       0.78      0.90      0.84       105
           1       0.82      0.64      0.72        74

    accuracy                           0.79       179
   macro avg       0.80      0.77      0.78       179
weighted avg       0.80      0.79      0.79       179

Prediction for new passenger: Did not survive


In [30]:
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier

titanic = fetch_openml(
    "titanic",
    version=1,
    as_frame=True
)
X = titanic.data[
    ["age", "fare", "embarked", "sex", "pclass"]
].copy()

y = titanic.target.astype(int)
X["age"] = pd.to_numeric(X["age"], errors="coerce")
X["fare"] = pd.to_numeric(X["fare"], errors="coerce")


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



#preprocessing
categorical_features = ["sex",'pclass','embarked']
numerical_features = ['age','fare']

categorical_pipeline = Pipeline(steps = [('imputer',SimpleImputer(strategy='most_frequent')),
                                          ('encoder',OneHotEncoder(handle_unknown='ignore'))
                                          ])

numerical_pipeline = Pipeline(steps = [('imputer',SimpleImputer(strategy='median')),
                                       ('scaler',StandardScaler())
                                       ])

preprocessor = ColumnTransformer(transformers = [("num",numerical_pipeline,numerical_features),
                                  ('cat',categorical_pipeline,categorical_features)
                                  ])

#machine learning model

full_pipeline = Pipeline(steps = [("preprocessing",preprocessor),
                          ("model",RandomForestClassifier(random_state = 42))
                          ])
#tune parameter
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__n_estimators":[100,200],
    "model__max_depth":[None,5],
    "model__min_samples_leaf":[1,5]
}

grid_search = GridSearchCV(
    estimator = full_pipeline,
    param_grid =param_grid,
    cv = 3,
    scoring= "accuracy",
    n_jobs=-1,
    refit= True
)

grid_search.fit(X_train,y_train)

print("Best Parameter",grid_search.best_params_)
print("Best accuracy",grid_search.best_score_)

#evaluation
from sklearn.metrics import  (accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

best_pipeline =grid_search.best_estimator_

y_pred = best_pipeline.predict(X_test)
y_prob = best_pipeline.predict_proba(X_test)[:, 1]

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test F1 Score:", f1_score(y_test, y_pred))
print("Test ROC-AUC:", roc_auc_score(y_test, y_prob))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Best Parameter {'model__max_depth': 5, 'model__min_samples_leaf': 1, 'model__n_estimators': 100}
Best accuracy 0.8051575931232092
Test Accuracy: 0.767175572519084
Test F1 Score: 0.680628272251309
Test ROC-AUC: 0.8559616290018831
Confusion Matrix:
 [[136   8]
 [ 53  65]]
